# Fine-Tuning Qwen2.5-Coder-32B for Solana Vulnerability Detection

This notebook trains Qwen2.5-Coder-32B-Instruct using QLoRA to detect security vulnerabilities in Solana smart contracts written in Rust.

The training approach follows the Chain-of-Thought (CoT) methodology, where the model learns to reason step-by-step before issuing a security verdict. This replicates and adapts the pipeline proposed by Tortora (2025) for Algorand, applying it to the Solana ecosystem.

**Why Qwen2.5-Coder-32B?**
- Code-specialized model trained on 5.5 trillion tokens across 92 programming languages
- 32B parameters (4x larger than LLaMA 3.1-8B used in Phase 1)
- State-of-the-art performance on code generation, reasoning, and repair benchmarks
- Provides a meaningful comparison axis: general-purpose small model vs. code-specialized large model

**Reference paper:** Boi, B. & Esposito, C. (2025). Prompt Engineering vs. Fine-Tuning for LLM-Based Vulnerability Detection in Solana and Algorand Smart Contracts. IEEE BCCA 2025.

**Hardware:** NVIDIA RTX A6000 (48GB VRAM), CUDA 12.6

## 1. Environment Verification

Verify that all required libraries are installed and the GPU is accessible.

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

import unsloth
print(f"Unsloth version: {unsloth.__version__}")

import trl
print(f"TRL version: {trl.__version__}")

PyTorch version: 2.10.0+cu126
CUDA available: True
GPU: NVIDIA RTX A6000
VRAM: 47.4 GB
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth version: 2026.4.6
TRL version: 0.24.0


## 2. Data Verification

Verify that the training and validation datasets are accessible from the cloned GitHub repository.

In [2]:
import os
import json

BASE_DIR = os.path.expanduser("~/NotebookProjects/bboi/Mustafa_LL")
DATA_DIR = os.path.join(BASE_DIR, "LLM-Contract-Analyzer", "data", "training")

TRAIN_PATH = os.path.join(DATA_DIR, "dataset_think_format.jsonl")
VAL_PATH = os.path.join(DATA_DIR, "validation_dataset.jsonl")

for path, name in [(TRAIN_PATH, "Training"), (VAL_PATH, "Validation")]:
    if os.path.exists(path):
        with open(path) as f:
            count = sum(1 for _ in f)
        print(f"{name} data: {count} samples")
    else:
        raise FileNotFoundError(f"{name} data not found at {path}")

Training data: 226 samples
Validation data: 22 samples


## 3. Data Preparation

The raw dataset stores the Chain-of-Thought reasoning and the final verdict in separate fields (`thinking` and `content`). For training with the model's native chat template, we merge them into a single assistant response wrapped in `<think>` and `<final>` tags.

This step also converts the `developer` role to `system`, which is the standard role name expected by the tokenizer.

In [3]:
WORK_DIR = os.path.join(BASE_DIR, "qwen_training")
os.makedirs(WORK_DIR, exist_ok=True)

def prepare_dataset(input_path, output_path):
    """Merge the separate thinking/content fields into a single CoT response."""
    prepared = []
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            sample = json.loads(line)
            new_messages = []

            for msg in sample["messages"]:
                if msg["role"] == "assistant" and "thinking" in msg:
                    merged = (
                        f"<think>\n{msg['thinking']}\n</think>\n\n"
                        f"<final>\n{msg['content']}\n</final>"
                    )
                    new_messages.append({"role": "assistant", "content": merged})
                elif msg["role"] == "developer":
                    new_messages.append({"role": "system", "content": msg["content"]})
                else:
                    new_messages.append(msg)

            prepared.append({"id": sample["id"], "messages": new_messages})

    with open(output_path, "w", encoding="utf-8") as f:
        for item in prepared:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    return len(prepared)

n_train = prepare_dataset(TRAIN_PATH, os.path.join(WORK_DIR, "train.jsonl"))
n_val = prepare_dataset(VAL_PATH, os.path.join(WORK_DIR, "val.jsonl"))

print(f"Training samples prepared: {n_train}")
print(f"Validation samples prepared: {n_val}")

# Quick sanity check on the merged format
with open(os.path.join(WORK_DIR, "train.jsonl")) as f:
    sample = json.loads(f.readline())
    response = sample["messages"][-1]["content"]
    assert "<think>" in response and "<final>" in response, "Format error"
    print(f"\nFormat verified: <think> and <final> tags present.")
    print(f"Sample response preview:\n{response[:300]}...")

Training samples prepared: 226
Validation samples prepared: 22

Format verified: <think> and <final> tags present.
Sample response preview:
<think>
Analyzing the Solana smart contract function `token_burn`.

1. **Function overview**: This function is part of a Solana program. I need to examine it for CPI error handling and result validation.

2. **Expected security checks for V6 (Unchecked Calls)**:
   - Check 1: CPI return value checke...


## 4. Load Base Model

We load Qwen2.5-Coder-32B-Instruct with 4-bit quantization (NF4). With 48GB VRAM on the RTX A6000, the quantized 32B model uses approximately 20GB, leaving ample room for training gradients and optimizer states.

Key difference from Phase 1: Qwen2.5-Coder is a code-specialized model trained on 5.5 trillion tokens across 92 programming languages, including Rust. This domain expertise should provide stronger baseline understanding of Solana contract patterns compared to the general-purpose LLaMA 3.1-8B.

In [4]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-32B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    full_finetuning=False,
)

print("Base model loaded: Qwen2.5-Coder-32B-Instruct (4-bit)")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.1f} GB")

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.4.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.422 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Base model loaded: Qwen2.5-Coder-32B-Instruct (4-bit)
GPU memory used: 17.9 GB
GPU memory reserved: 18.0 GB


## 5. LoRA Adapter Configuration

We use Low-Rank Adaptation (LoRA) to efficiently fine-tune the model. Instead of updating all 32 billion parameters, LoRA freezes the original weights and injects small trainable matrices.

The configuration matches Phase 1 (LLaMA) and follows Tortora (2025), Section 4.4.1:
- Rank r = 64 (captures complex vulnerability patterns)
- Alpha = 128 (scaling factor, 2x the rank)
- Target: all linear layers (attention + MLP)

This ensures a fair comparison between the two models, isolating the effect of model architecture and size.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
print(f"GPU memory after LoRA: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

Unsloth 2026.4.6 patched 64 layers with 64 QKV layers, 64 O layers and 64 MLP layers.


Trainable parameters: 536,870,912 / 33,300,747,264 (1.61%)
GPU memory after LoRA: 20.0 GB


## 6. Load Datasets

Load the prepared training and validation sets using the Hugging Face `datasets` library.

In [6]:
from datasets import load_dataset

dataset_train = load_dataset(
    "json",
    data_files=os.path.join(WORK_DIR, "train.jsonl"),
    split="train",
)

dataset_eval = load_dataset(
    "json",
    data_files=os.path.join(WORK_DIR, "val.jsonl"),
    split="train",
)

print(f"Training samples: {len(dataset_train)}")
print(f"Validation samples: {len(dataset_eval)}")

# Verify the chat template format with one sample
sample_msgs = dataset_train[0]["messages"]
formatted = tokenizer.apply_chat_template(sample_msgs, tokenize=False)
print(f"\nFormatted sample preview (first 500 chars):\n{formatted[:500]}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Training samples: 226
Validation samples: 22

Formatted sample preview (first 500 chars):
<|im_start|>system
You are an expert smart contract security auditor specialized in the Solana blockchain and Rust.
Your task is to analyze Rust code precisely and systematically to identify security vulnerabilities.

### Required behavior:
- Always perform a structured, explicit reasoning phase first and put it inside the <think> block.
  - In <think>...</think> you must:
    - Summarize the contract's purpose and high-level architecture.
    - Inspect the logic block-by-block (or line-by-line 


## 7. Training Configuration

All hyperparameters match Phase 1 (LLaMA 3.1-8B) exactly to ensure a fair comparison:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Epochs | 3 | Same as Phase 1 and Tortora (2025) |
| Effective batch size | 8 (1 x 8 accumulation) | Same as Phase 1 |
| Learning rate | 5e-5 | Cosine decay with 5% warmup |
| Optimizer | Paged AdamW 8-bit | Memory-efficient variant |
| Max sequence length | 2048 tokens | Same as Phase 1 |

The `train_on_responses_only` strategy ensures the loss is computed only on the assistant's output (CoT reasoning + verdict), not on the system prompt or input code.

**Note:** Qwen2.5 uses the ChatML format with `<|im_start|>` and `<|im_end|>` tokens, which differs from LLaMA's format. The response masking tokens are adjusted accordingly.

In [8]:
from trl import SFTConfig, SFTTrainer
from unsloth import train_on_responses_only

MODEL_OUTPUT_DIR = os.path.join(BASE_DIR, "models", "Qwen2.5-Coder-32B-Solana-Audit")

# Pre-format the datasets: convert messages to text using the chat template
def convert_to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset_train_fmt = dataset_train.map(convert_to_text, remove_columns=dataset_train.column_names)
dataset_eval_fmt = dataset_eval.map(convert_to_text, remove_columns=dataset_eval.column_names)

print(f"Training samples formatted: {len(dataset_train_fmt)}")
print(f"Sample preview:\n{dataset_train_fmt[0]['text'][:300]}...")

sft_config = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,

    num_train_epochs=3,
    max_seq_length=MAX_SEQ_LENGTH,

    learning_rate=5e-5,
    warmup_steps=4,
    lr_scheduler_type="cosine",

    weight_decay=0.02,
    optim="paged_adamw_8bit",
    max_grad_norm=1.0,

    eval_strategy="steps",
    eval_steps=10,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=3,

    dataset_text_field="text",

    seed=3407,
    output_dir=MODEL_OUTPUT_DIR,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train_fmt,
    eval_dataset=dataset_eval_fmt,
    args=sft_config,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("Trainer configured. Ready to start.")
print(f"Output directory: {MODEL_OUTPUT_DIR}")

Map:   0%|          | 0/226 [00:00<?, ? examples/s]

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

Training samples formatted: 226
Sample preview:
<|im_start|>system
You are an expert smart contract security auditor specialized in the Solana blockchain and Rust.
Your task is to analyze Rust code precisely and systematically to identify security vulnerabilities.

### Required behavior:
- Always perform a structured, explicit reasoning phase fir...


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/226 [00:00<?, ? examples/s]

num_proc must be <= 22. Reducing num_proc to 22 for dataset of size 22.
[datasets.arrow_dataset|WARNING]num_proc must be <= 22. Reducing num_proc to 22 for dataset of size 22.


...

num_proc must be <= 22. Reducing num_proc to 22 for dataset of size 22.
[datasets.arrow_dataset|WARNING]num_proc must be <= 22. Reducing num_proc to 22 for dataset of size 22.


Map (num_proc=22):   0%|          | 0/22 [00:00<?, ? examples/s]

num_proc must be <= 22. Reducing num_proc to 22 for dataset of size 22.
[datasets.arrow_dataset|WARNING]num_proc must be <= 22. Reducing num_proc to 22 for dataset of size 22.


Filter (num_proc=22):   0%|          | 0/22 [00:00<?, ? examples/s]

Trainer configured. Ready to start.
Output directory: /home/habes/NotebookProjects/bboi/Mustafa_LL/models/Qwen2.5-Coder-32B-Solana-Audit


## 8. Training

This cell runs the fine-tuning. On the RTX A6000 (48GB), expect approximately 1-2 hours for 3 epochs over 226 samples with a 32B model.

Monitor:
- **train_loss**: should decrease steadily (the model is learning)
- **eval_loss**: should decrease then stabilize (generalization check)

In [9]:
trainer_stats = trainer.train()

print(f"\nTraining completed in {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"({trainer_stats.metrics['train_runtime'] / 60:.1f} minutes)")
print(f"Final training loss: {trainer_stats.metrics['train_loss']:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 226 | Num Epochs = 3 | Total steps = 87
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 536,870,912 of 33,300,747,264 (1.61% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,1.771001,1.288708
20,0.863970,0.454816
30,0.340865,0.268979
40,0.249051,0.220174
50,0.211322,0.194492
60,0.198734,0.180369
70,0.180595,0.172835
80,0.173463,0.170195
87,0.173463,0.170065


/home/habes/anaconda3/envs/biagio_8890/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/habes/anaconda3/envs/biagio_8890/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/habes/anaconda3/envs/biagio_8890/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverte


Training completed in 1794 seconds
(29.9 minutes)
Final training loss: 0.4722


## 9. Save the Trained Adapter

We save only the LoRA adapter weights, not the full 32B model. The adapter is typically 200-400MB compared to the ~64GB full model.

In [10]:
model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

total_size = sum(
    os.path.getsize(os.path.join(MODEL_OUTPUT_DIR, f))
    for f in os.listdir(MODEL_OUTPUT_DIR)
    if os.path.isfile(os.path.join(MODEL_OUTPUT_DIR, f))
) / (1024 * 1024)

print(f"Adapter saved to {MODEL_OUTPUT_DIR}")
print(f"Total size: {total_size:.1f} MB")

Adapter saved to /home/habes/NotebookProjects/bboi/Mustafa_LL/models/Qwen2.5-Coder-32B-Solana-Audit
Total size: 2059.0 MB


## 10. Inference Test

A quick sanity check to verify the fine-tuned model produces structured output with `<think>` and `<final>` tags. We test on a Solana function with a deliberate vulnerability (missing signer verification).

In [11]:
FastLanguageModel.for_inference(model)

test_code = """
pub fn process_withdraw(
    program_id: &Pubkey,
    accounts: &[AccountInfo],
    amount: u64,
) -> ProgramResult {
    let authority_info = next_account_info(account_info_iter)?;
    let vault_info = next_account_info(account_info_iter)?;
    let destination_info = next_account_info(account_info_iter)?;

    let transfer_ix = spl_token::instruction::transfer(
        token_program.key, vault_info.key,
        destination_info.key, authority_info.key,
        &[], amount,
    )?;
    invoke(&transfer_ix, &[vault_info.clone(), destination_info.clone(), authority_info.clone()])?;
    Ok(())
}
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are an expert smart contract security auditor specialized in "
            "the Solana blockchain and Rust. Analyze the code for vulnerabilities. "
            "Put your reasoning inside <think> tags and your verdict inside <final> tags."
        ),
    },
    {
        "role": "user",
        "content": f"Analyze this Solana contract for vulnerabilities:\n\n{test_code}",
    },
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs, max_new_tokens=512, temperature=0.6, top_p=0.95
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("Model response:")
print("=" * 60)
print(response)
print("=" * 60)
print(f"\nContains <think>: {'<think>' in response}")
print(f"Contains <final>: {'<final>' in response}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/habes/anaconda3/envs/biagio_8890/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/habes/anaconda3/envs/biagio_8890/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: T

Model response:
<think>
Analyzing the Solana smart contract function `process_withdraw`.

1. **Function overview**: This function is part of a Solana program. I need to examine it for CPI error handling and result validation.

2. **Expected security checks for V6 (Unchecked Calls)**:
   - Check 1: CPI return value checked (no `let _ =` or ignored Result)
   - Check 2: error propagation with `?` operator on invoke/invoke_signed
   - Check 3: validation of returned data from external programs

3. **Code inspection**: Examining the implementation of `process_withdraw`:
   - Both instruction creation and invocation use ? for error propagation. If either fails, the entire withdraw operation aborts.

4. **Assessment**: The code correctly implements the required security measures. The expected checks for V6 are present and properly enforced.

5. **Conclusion**: No vulnerability detected. The contract handles CPI error handling and result validation appropriately.
</think>

<final>NO_VULNERABI

## 11. Push to Hugging Face Hub

Upload the trained LoRA adapter to Hugging Face for later use in the evaluation phase.

In [12]:
# Replace with your Hugging Face token
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
HF_REPO = "Mustafa99Hafed/Qwen2.5-Coder-32B-Solana-Audit"

model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method="lora",
    token=HF_TOKEN,
)

print(f"Adapter pushed to: https://huggingface.co/{HF_REPO}")
print("Done!")

config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-69e8cdbe-278be4371624d3cb0f74e189;b3a4dd3d-ec21-499b-a055-54808bed110c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.